# 🌍 ESG RL Training — GRPO + Unsloth

**OpenEnv Hackathon | ESG Compliance & Sustainability Environment**

Trains an LLM agent with **Group Relative Policy Optimization (GRPO)** using verifiable ESG environment rewards.

**Stack:** Unsloth (4-bit LoRA) · TRL (GRPOTrainer) · ESGEnvironment (verifiable reward)

**Runtime:** T4 GPU recommended (Google Colab free tier works)

| Cell | Purpose |
|------|---------|
| 1 | Setup & GPU check |
| 2 | Install dependencies |
| 3 | Clone repo |
| 4 | Smoke test (validate pipeline without GPU) |
| 5 | Generate training dataset |
| 6 | Run GRPO training |
| 7 | Benchmark & plot results |
| 8 | (Optional) Push model to HuggingFace Hub |

In [ ]:
# Cell 1 — Setup & GPU check
import subprocess, sys

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'No GPU detected — training will use CPU (slow)')

try:
    import torch
    print(f'PyTorch: {torch.__version__}')
    print(f'CUDA available: {torch.cuda.is_available()}')
    if torch.cuda.is_available():
        print(f'GPU: {torch.cuda.get_device_name(0)}')
        print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
except ImportError:
    print('PyTorch not yet installed — will be installed in Cell 2')

In [ ]:
# Cell 2 — Install dependencies
# Unsloth for fast 4-bit training, TRL for GRPO, and project dependencies
!pip install -q pydantic openai pyyaml datasets

# Unsloth + TRL (GPU path)
import subprocess, sys
result = subprocess.run(['nvidia-smi'], capture_output=True)
HAS_GPU = result.returncode == 0

if HAS_GPU:
    !pip install -q 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'
    !pip install -q trl peft accelerate bitsandbytes
    print('Installed: Unsloth + TRL (GPU mode)')
else:
    !pip install -q trl peft transformers accelerate
    print('Installed: TRL + HuggingFace (CPU fallback mode)')

print('All dependencies installed.')

In [ ]:
# Cell 3 — Clone repository
# Option A: GitHub (update URL to your repo)
GITHUB_REPO = 'https://github.com/TharunBabu-05/OPEN-ENV.git'
REPO_DIR   = '/content/open_ENV'

import os
if not os.path.exists(REPO_DIR):
    !git clone {GITHUB_REPO} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
    print('Repository already present — pulled latest.')

%cd {REPO_DIR}
!python -c "from env import ESGEnvironment; from tasks import TASKS; print('Import check OK — tasks:', list(TASKS.keys()))"

# Option B: Google Drive (uncomment if preferred)
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/open_ENV

In [ ]:
# Cell 4 — Smoke test (no GPU, no model download needed)
# Validates: dataset builder, reward functions, env stepping.
# Expected output: "SMOKE TEST PASSED [OK]"
!python train_rl.py --smoke_test

In [ ]:
# Cell 5 — Generate training dataset
# Rolls out the ESG environment with a heuristic policy to create
# a diverse prompt dataset for GRPO training.
!python dataset_builder.py --episodes 20 --output data/esg_prompts.jsonl

# Show dataset stats
with open('data/esg_prompts.jsonl') as f:
    lines = f.readlines()
import json
sample = json.loads(lines[0])
print(f'\nDataset size:   {len(lines)} prompts')
print(f'Tasks covered:  {set(json.loads(l)["task_id"] for l in lines)}')
print(f'Sample task_id: {sample["task_id"]} | step: {sample["step"]}')

In [ ]:
# Cell 6 — GRPO training
# Start with the easy task (basic_compliance, 6 steps) to ensure
# non-zero rewards early in training — follow curriculum §6.
#
# Adjust --max_steps based on available time:
#   T4 Colab (~15 min):  --max_steps 200
#   A100 Colab (~5 min): --max_steps 500
#   CPU only (debug):    --max_steps 5 --cpu_only

!python train_rl.py \
    --config train_config.yaml \
    --task basic_compliance \
    --max_steps 200 \
    --output_dir outputs/esg_rl_v1

import os
adapter = 'outputs/esg_rl_v1/lora_adapter'
if os.path.exists(adapter):
    print(f'\nAdapter saved -> {adapter}')
    print('Files:', os.listdir(adapter))
else:
    print('WARNING: adapter not found — check training logs above.')

In [ ]:
# Cell 7 — Benchmark & visualize results
# Runs heuristic baseline + (if adapter exists) trained model,
# then generates 3 comparison plots.
import os

# Always run heuristic baseline
!python benchmark.py --mode heuristic --seeds 42 43 44 --output results/baseline_heuristic.json

# Run trained model if adapter exists
adapter = 'outputs/esg_rl_v1/lora_adapter'
if os.path.exists(adapter):
    !python benchmark.py --mode llm --model_path {adapter} --seeds 42 43 44 --output results/trained_v1.json
    !python plot_results.py --baseline results/baseline_heuristic.json --trained results/trained_v1.json --output_dir results
else:
    !python plot_results.py --baseline results/baseline_heuristic.json --output_dir results

# Display plots inline
from IPython.display import Image, display
for plot in ['score_comparison.png', 'reward_history.png', 'esg_metrics.png']:
    path = f'results/{plot}'
    if os.path.exists(path):
        print(f'\n--- {plot} ---')
        display(Image(path))

In [ ]:
# Cell 8 — (Optional) Push model to HuggingFace Hub
# Uncomment and fill in your HF token + username when credits are available.

# from huggingface_hub import login, HfApi
# HF_TOKEN    = 'hf_...'          # Your HuggingFace token
# HF_USERNAME = 'YOUR_USERNAME'   # Your HuggingFace username
# MODEL_NAME  = 'esg-rl-agent'
#
# login(token=HF_TOKEN)
#
# # Push adapter
# from transformers import AutoTokenizer
# from peft import PeftModel
# tokenizer = AutoTokenizer.from_pretrained('outputs/esg_rl_v1/lora_adapter')
# tokenizer.push_to_hub(f'{HF_USERNAME}/{MODEL_NAME}', token=HF_TOKEN)
#
# api = HfApi(token=HF_TOKEN)
# api.upload_folder(
#     folder_path='outputs/esg_rl_v1/lora_adapter',
#     repo_id=f'{HF_USERNAME}/{MODEL_NAME}',
#     repo_type='model',
# )
# print(f'Model pushed -> https://huggingface.co/{HF_USERNAME}/{MODEL_NAME}')

print('Cell 8: Uncomment the lines above and add your HF token to push the trained model.')